# Unità 1 — BB84 ideale

In questo notebook simuliamo il protocollo BB84 nel caso ideale, cioè senza Eve e senza rumore.

## Setup e import

Prepariamo il percorso del progetto in modo da poter importare le funzioni dalla cartella `src`.

In [1]:
from pathlib import Path
import sys

current_path = Path.cwd()

if (current_path / "src" / "bb84.py").exists():
    project_root = current_path
elif (current_path.parent / "src" / "bb84.py").exists():
    project_root = current_path.parent
else:
    raise FileNotFoundError("Non trovo la cartella src del progetto.")

src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

In [2]:
import pandas as pd
from qiskit import QuantumCircuit

from bb84 import (
    random_bits,
    random_bases,
    prepare_bb84_state,
    measure_bb84_state,
    run_bb84_round,
    run_bb84_protocol,
    sift_keys,
    compute_qber,
)

## Generazione casuale di bit e basi

Alice genera bit casuali e sceglie basi casuali. Anche Bob sceglie basi casuali per misurare i qubit ricevuti.

In [3]:
n = 10

alice_bits = random_bits(n, seed=1)
alice_bases = random_bases(n, seed=2)
bob_bases = random_bases(n, seed=3)

print("Bit di Alice:", alice_bits)
print("Basi di Alice:", alice_bases)
print("Basi di Bob:", bob_bases)

Bit di Alice: [0, 1, 1, 1, 0, 0, 1, 1, 0, 0]
Basi di Alice: ['X', 'Z', 'Z', 'Z', 'Z', 'X', 'Z', 'Z', 'Z', 'X']
Basi di Bob: ['X', 'Z', 'Z', 'Z', 'Z', 'X', 'X', 'X', 'Z', 'Z']


## Preparazione degli stati BB84

La preparazione usa questa corrispondenza:

bit 0, base Z -> |0>

bit 1, base Z -> |1>

bit 0, base X -> |+>

bit 1, base X -> |->

In [4]:
states = [
    (0, "Z"),
    (1, "Z"),
    (0, "X"),
    (1, "X"),
]

for bit, basis in states:
    circuit = QuantumCircuit(1)
    prepare_bb84_state(circuit, bit, basis)
    print("bit =", bit, "base =", basis)
    print(circuit.draw())
    print()

bit = 0 base = Z
   
q: 
   

bit = 1 base = Z
   ┌───┐
q: ┤ X ├
   └───┘

bit = 0 base = X
   ┌───┐
q: ┤ H ├
   └───┘

bit = 1 base = X
   ┌───┐┌───┐
q: ┤ X ├┤ H ├
   └───┘└───┘



## Misura in base Z e in base X

Bob misura direttamente se usa la base Z. Se invece usa la base X, applica prima un gate H e poi misura.

In [5]:
circuit = QuantumCircuit(1, 1)
prepare_bb84_state(circuit, bit=0, basis="X")
measure_bb84_state(circuit, basis="X")

print(circuit.draw())

     ┌───┐┌───┐┌─┐
  q: ┤ H ├┤ H ├┤M├
     └───┘└───┘└╥┘
c: 1/═══════════╩═
                0 


## Singolo round BB84

Un round BB84 ideale contiene la preparazione del qubit da parte di Alice, la misura da parte di Bob e il confronto successivo delle basi.

In [6]:
bob_bit = run_bb84_round(alice_bit=1, alice_basis="Z", bob_basis="Z")
print("Bit ottenuto da Bob nel primo round:", bob_bit)

bob_bit = run_bb84_round(alice_bit=0, alice_basis="X", bob_basis="X")
print("Bit ottenuto da Bob nel secondo round:", bob_bit)

Bit ottenuto da Bob nel primo round: 1
Bit ottenuto da Bob nel secondo round: 0


## Protocollo multi-round

Ora ripetiamo il round più volte. Per ogni round salviamo il bit di Alice, la base di Alice, la base di Bob, il bit misurato da Bob e l'informazione `keep`.

In [19]:
results = run_bb84_protocol(n_rounds=100, seed=115)

for i in range(5):
    print(results[i])

{'round': 1, 'alice_bit': 1, 'alice_basis': 'Z', 'bob_basis': 'Z', 'bob_bit': 1, 'keep': True}
{'round': 2, 'alice_bit': 1, 'alice_basis': 'Z', 'bob_basis': 'X', 'bob_bit': 0, 'keep': False}
{'round': 3, 'alice_bit': 0, 'alice_basis': 'X', 'bob_basis': 'Z', 'bob_bit': 0, 'keep': False}
{'round': 4, 'alice_bit': 1, 'alice_basis': 'X', 'bob_basis': 'Z', 'bob_bit': 1, 'keep': False}
{'round': 5, 'alice_bit': 0, 'alice_basis': 'Z', 'bob_basis': 'Z', 'bob_bit': 0, 'keep': True}


## Sifting

Il sifting conserva solo i round in cui Alice e Bob hanno scelto la stessa base.

In [20]:
alice_key, bob_key = sift_keys(results)

print("Chiave di Alice:", alice_key)
print("Chiave di Bob:", bob_key)
print("Lunghezza della chiave sifted:", len(alice_key))

Chiave di Alice: [1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1]
Chiave di Bob: [1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1]
Lunghezza della chiave sifted: 49


## Calcolo del QBER

Il QBER misura la frazione di bit diversi tra la chiave di Alice e la chiave di Bob dopo il sifting.

In [21]:
qber = compute_qber(alice_key, bob_key)

print("QBER:", qber)

if qber == 0:
    print("Nel caso ideale il QBER è nullo.")
else:
    print("È stato ottenuto un QBER non nullo: controllare la simulazione.")

QBER: 0.0
Nel caso ideale il QBER è nullo.


## Rappresentazione tabellare dei risultati

La simulazione restituisce una lista di dizionari. Per visualizzare meglio i risultati, possiamo convertirla in una tabella pandas. Questa è solo una fase di rappresentazione, separata dalla logica del protocollo.

In [22]:
df = pd.DataFrame(results)
df.head(10)

,round,alice_bit,alice_basis,bob_basis,bob_bit,keep
0,1,1,Z,Z,1,True
1,2,1,Z,X,0,False
2,3,0,X,Z,0,False
3,4,1,X,Z,1,False
4,5,0,Z,Z,0,True
5,6,1,X,X,1,True
6,7,0,X,X,0,True
7,8,1,X,Z,1,False
8,9,0,Z,X,0,False
9,10,0,X,X,0,True


## Commento finale

Nel caso ideale, dopo il sifting, Alice e Bob ottengono la stessa chiave. Di conseguenza il QBER è pari a 0. Questo notebook rappresenta il caso di riferimento rispetto al quale saranno confrontati l'attacco intercept-resend di Eve e, successivamente, gli effetti del rumore.